# Phase 4: Network-Wide Multivariate Tabular Baselines
### High-Dimensional Linear Regression & Random Forest across 207 Sensors

This notebook benchmarks classical machine learning models on the full network:
1. Ingesting METR-LA dataset.
2. Flattening 12 historical timesteps across all 207 sensors into $12 \times 207 = 2,484$ lag features.
3. Chronological time-series partition: 70% Train, 15% Validation, 15% Test.
4. Training and evaluating Multivariate Linear Regression (OLS).
5. Training and evaluating Multivariate Random Forest Regressor.


## Section 1: Environment Setup, Dependencies & System Configuration
We configure the runtime environment, set plotting styles, verify GPU hardware acceleration, and import fundamental scientific computing libraries (`numpy`, `pandas`, `matplotlib`).


In [4]:
#Day01_Traffic_Exploration.ipynb
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

print("Setup complete ")


Setup complete 


In [ ]:
# Ingest METR-LA dataset (checks local directory or Colab environment)
import os
import zipfile
import pandas as pd

file_path = 'METR-LA.h5'
zip_path = 'METR-LA.h5.zip'

if not os.path.exists(file_path):
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
    elif os.path.exists('/content/METR-LA.h5.zip'):
        !unzip -o /content/METR-LA.h5.zip -d /content/
        file_path = '/content/METR-LA.h5'

if os.path.exists(file_path):
    df = pd.read_hdf(file_path)
elif os.path.exists('/content/METR-LA.h5'):
    df = pd.read_hdf('/content/METR-LA.h5')
else:
    # Fallback to local or downloaded copy
    df = pd.read_hdf('METR-LA.h5')

print("Dataset Shape (timesteps, sensors):", df.shape)
df.head()


## Section 8: Multivariate Tabular Baselines (Linear Regression & Random Forest across 207 Sensors)
To benchmark against traditional machine learning approaches that ignore graph structure, we flatten the 12 past timesteps across all 207 sensors into a high-dimensional feature vector ($12 \times 207 = 2,484$ input features) to predict the next speed reading for all 207 sensors simultaneously.

### 8.1 Multivariate Sequence Extraction & Chronological Splitting (70% Train, 15% Val, 15% Test)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---- 1. Build lag features for ALL sensors at once ----
SEQ_LENGTH = 12  # 12 steps * 5 min = 1 hour of history

def create_multivariate_sequences(data, seq_length):
    """
    data: (timesteps, num_sensors)
    returns X: (num_samples, seq_length * num_sensors), y: (num_samples, num_sensors)
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length].flatten())   # flatten lags across all sensors
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

values = df.values  # (34272, 207) — your reconstructed speed matrix
X, y = create_multivariate_sequences(values, SEQ_LENGTH)
print("X shape:", X.shape)   # (34260, 12*207)
print("y shape:", y.shape)   # (34260, 207)

# ---- 2. Time-based split (no shuffling — it's a time series) ----
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val     = X[train_end:val_end], y[train_end:val_end]
X_test, y_test   = X[val_end:], y[val_end:]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


### 8.2 Train & Evaluate Multivariate Linear Regression and Random Forest


In [ ]:
# ---- 3. Linear Regression baseline ----
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
print(f"Linear Regression -> MAE: {mae_lr:.3f}, RMSE: {rmse_lr:.3f}")

# ---- 4. Random Forest baseline ----
# NOTE: RF on 12*207=2484 input features and 207 outputs is heavy.
# Start with fewer trees / limited depth so it finishes in reasonable time.
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
print(f"Random Forest -> MAE: {mae_rf:.3f}, RMSE: {rmse_rf:.3f}")


X shape: (34260, 2484)
y shape: (34260, 207)
Train: (23982, 2484) Val: (5139, 2484) Test: (5139, 2484)
Linear Regression -> MAE: 4.013, RMSE: 7.077
